# 07 — Misclassification analysis (Figure 7)

**Tujuan.** Siapkan metadata kesalahan dari final-test JSON yang sudah dibekukan, tanpa inference ulang. Figure 7 hlm. 91512–91513 menunjukkan DDR DR→No DR (FN) dan No DR→DR (FP); ini semantik biner, berbeda dari salah grade lima kelas. Gambar test **tidak dibuka** saat readiness.

In [1]:
from pathlib import Path
import os, sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "reproduction/cadvit/model.py").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
ARTIFACT_ROOT = Path(os.environ["CAD_VIT_ARTIFACT_ROOT"]).expanduser().resolve() if os.environ.get("CAD_VIT_ARTIFACT_ROOT") else None
print("Artifact root:", ARTIFACT_ROOT if ARTIFACT_ROOT else "PENDING (set CAD_VIT_ARTIFACT_ROOT)")

Artifact root: PENDING (set CAD_VIT_ARTIFACT_ROOT)


In [2]:
from reproduction.cadvit.visualization import load_controlled_result, error_records
if ARTIFACT_ROOT:
    item = load_controlled_result(ARTIFACT_ROOT, "balanced_v1", "ddr_full")
    print("DDR", item["status"])
    if item["status"] == "AVAILABLE":
        errors = error_records(item["result"])
        print(errors["status"], "error records", len(errors.get("errors", [])),
              "task", errors.get("task"))
        binary = error_records(item["result"], binary_projection=True)
        print("binary DR/No DR projection (ASSUMED)", binary["status"],
              "FP/FN records", len(binary.get("errors", [])))
        print("Gallery PENDING: explicit post-evaluation image opt-in required")
else:
    print("PENDING: frozen per-image prediction metadata unavailable")

PENDING: frozen per-image prediction metadata unavailable


In [3]:
from reproduction.cadvit.visualization import plot_error_gallery, save_campaign_figure
if os.environ.get("CAD_VIT_SHOW_TEST_GALLERY") == "1":
    from IPython.display import display
    if ARTIFACT_ROOT is None:
        raise ValueError("CAD_VIT_ARTIFACT_ROOT required")
    item = load_controlled_result(ARTIFACT_ROOT, "balanced_v1", "ddr_full")
    if item["status"] != "AVAILABLE":
        raise ValueError("Frozen DDR final-test JSON unavailable or unverified")
    rows = error_records(item["result"], binary_projection=True)
    if rows["status"] != "AVAILABLE":
        raise ValueError("Per-image prediction metadata unavailable")
    dataset_root = os.environ["CAD_VIT_DATA_ROOT"]
    figure = plot_error_gallery(rows["errors"], dataset_root, allow_test_images=True)
    display(figure)
    if os.environ.get("CAD_VIT_SAVE_FIGURES") == "1":
        print("Saved", save_campaign_figure(figure, ARTIFACT_ROOT,
                                             "balanced_v1", "ddr_full", "error_gallery"))
else:
    print("PENDING gallery: test images remain unopened; opt in only after final evaluation is frozen")

PENDING gallery: test images remain unopened; opt in only after final evaluation is frozen


**Interpretasi dan keterbatasan.** Runner baru menyimpan image ID, label aktual/prediksi, confidence, serta SHA-256 di hasil final test **hanya ketika perintah test eksplisit kelak dijalankan**. `plot_error_gallery(..., allow_test_images=True)` tersedia untuk tahap pelaporan setelah evaluasi dikunci; jangan memilih contoh berdasarkan test untuk training/tuning. JSON historis tanpa prediksi per gambar tetap PENDING.
**Implementation assumption:** Galeri FP/FN memproyeksikan grade 0 ke No DR dan grade 1–4 ke DR. Kesalahan grade lima kelas tetap dilaporkan terpisah; proyeksi ini tidak diklaim sebagai protokol Figure 7 penulis.
